# Run Model

Single-country simulation runner with standard macro, benchmark, permanent-income, sector, and firm-credit diagnostics. Configure the inputs, run the workflow once, then inspect the manifest and diagnostic sections below.

## Companion notebooks

- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.monte_carlo import run_seeded_monte_carlo
from config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    plot_firm_credit_to_equity_and_capital,
    plot_mc,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)

## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [2]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"
# SCENARIO_NAME = "firm_dividend_payout_ratio"

run_config = NotebookRunConfig(
    seed=40,
    t_max=50,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,  # unmodified France config: fixed alpha=0
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]

## Run simulation

In [3]:
# In the first run, state is none. After, run_notebook_workflow uses prepared from the previous state, preventing reloading it.
state = globals().get("state")

state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
    previous_state=state,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_scenario = state.df_scenario
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

{'seed': 40, 'country': 'FRA', 't_max': 50, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                              

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12 (+2.84%), industry 14 (+4.92%), industry 15 (+1.99%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0.75,
                                  'individuals_quitting': False,
                                  'individuals_quitting_temperature': 1.0,
                                  'optimised_hirin

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12 (+2.84%), industry 14 (+4.92%), industry 15 (+1.99%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()


Simulation complete. Output saved to: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_benchmark.h5
Benchmark dataframe cached at: /Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/FRA_df_benchmark.pkl


In [4]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


{'country': 'FRA',
 'seed': 40,
 't_max': 50,
 'scenario': 'calibrated_consumption',
 'model_h5': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5',
 'benchmark': True,
 'manifest': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/run_manifest_FRA_seed40_t50.json'}

## Macro, fiscal, policy, labour, and balance sheets

In [5]:
plot_output(
    df=df_scenario[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_scenario[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


In [6]:
from src.visual_helpers import plot_agent_timeseries  # noqa: E402


hh_balance_sheet_vars = [
    "liquid_financial_assets",
    "illiquid_financial_assets",
    "wealth_main_residence",
    "wealth_other_properties",  # other properties, not main residence
    "wealth_other_real_assets", # other real assets, not properties
    "wealth",  # gross wealth
    "mortgage_debt",
    "consumption_loan_debt",
    "debt",
    "net_wealth",  # wealth after liabilities
]

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    hh_balance_sheet_vars,
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    # no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

## Benchmark comparison

In [7]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_scenario[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Fiscal diagnostics

In [8]:
def fiscal_diagnostic_table(df, items, periods=(1, 20, 50), unit=1e9, currency="€"):
    """Amount (bn) and % of GDP for each fiscal line item, at the given periods."""
    rows = {}
    for label, col in items.items():
        ratio_col = f"{col}_to_gdp"
        row = {}
        for t in periods:
            amount = df.loc[t, col] / unit
            pct = df.loc[t, ratio_col] * 100 if ratio_col in df.columns else float("nan")
            row[f"Period {t}"] = f"{currency}{amount:,.3f}bn ({pct:.2f}%)"
        rows[label] = row
    return pd.DataFrame(rows).T


REVENUE_ITEMS = {
    "Corporate income tax": "fiscal_revenue_corporate_income_taxes",
    "Household income tax": "fiscal_revenue_income_taxes",
    "Production tax": "fiscal_revenue_production_taxes",
    "VAT": "fiscal_revenue_vat",
    "Capital-formation taxes": "fiscal_revenue_capital_formation_taxes",
    "Export taxes": "fiscal_revenue_export_taxes",
    "Employee social insurance": "fiscal_revenue_employee_social_insurance",
    "Employer social insurance": "fiscal_revenue_employer_social_insurance",
    "Total revenue": "fiscal_revenue",
}

EXPENDITURE_ITEMS = {
    "Government consumption": "government_consumption",
    "Unemployment benefits": "unemployment_benefits",
    "Interest payments on debt": "interest_payments_on_debt",
    "Household social transfers": "household_social_transfers",
    "Public pension benefits": "public_pension_benefits",
    "Other social transfers": "other_social_transfers",
    "Necessity support": "necessity_support",
    "Total expenditure": "fiscal_expenditure",
}


from IPython.display import Markdown

fiscal_periods = (1, 20, 50)
revenue_table = fiscal_diagnostic_table(df_scenario, REVENUE_ITEMS, periods=fiscal_periods)
expenditure_table = fiscal_diagnostic_table(df_scenario, EXPENDITURE_ITEMS, periods=fiscal_periods)
print("Scenario tables")
display(Markdown("**Revenue**\n\n" + revenue_table.to_markdown()))
display(Markdown("**Expenditure**\n\n" + expenditure_table.to_markdown()))


REVENUES_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "fiscal_revenue_vat_to_gdp",
    "fiscal_revenue_production_taxes_to_gdp",
    "fiscal_revenue_capital_formation_taxes_to_gdp",
    "fiscal_revenue_corporate_income_taxes_to_gdp",
    "fiscal_revenue_income_taxes_to_gdp",
    "fiscal_revenue_rental_income_taxes_to_gdp",
    "fiscal_revenue_employee_social_insurance_to_gdp",
    "fiscal_revenue_employer_social_insurance_to_gdp",
    "fiscal_revenue_taxes_on_products_to_gdp",
    "fiscal_revenue_export_taxes_to_gdp",
    "fiscal_revenue_social_housing_rent_to_gdp",

    "fiscal_revenue_vat",
    "fiscal_revenue_production_taxes",
    "fiscal_revenue_capital_formation_taxes",
    "fiscal_revenue_corporate_income_taxes",
    "fiscal_revenue_income_taxes",
    "fiscal_revenue_rental_income_taxes",
    "fiscal_revenue_employee_social_insurance",
    "fiscal_revenue_employer_social_insurance",
    "fiscal_revenue_taxes_on_products",
    "fiscal_revenue_export_taxes",
    "fiscal_revenue_social_housing_rent",
    ]

EXPENDITURE_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "government_consumption_to_gdp",
    "unemployment_benefits_to_gdp",
    "interest_payments_on_debt_to_gdp",
    "household_social_transfers_to_gdp",
    "public_pension_benefits_to_gdp",
    "other_social_transfers_to_gdp",
    "necessity_support_to_gdp",

    "government_consumption",
    "unemployment_benefits",
    "interest_payments_on_debt",
    "household_social_transfers",
    "public_pension_benefits",
    "other_social_transfers",
    "necessity_support",
]

plot_output(df=df_scenario[list(REVENUES_COLS)], no_rows=7, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(EXPENDITURE_COLS)], no_rows=6, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


Scenario tables


**Revenue**

|                           | Period 1            | Period 20           | Period 50           |
|:--------------------------|:--------------------|:--------------------|:--------------------|
| Corporate income tax      | €29.461bn (5.76%)   | €22.136bn (4.28%)   | €38.056bn (5.84%)   |
| Household income tax      | €43.650bn (8.53%)   | €39.273bn (7.60%)   | €42.661bn (6.54%)   |
| Production tax            | €16.566bn (3.24%)   | €16.204bn (3.14%)   | €20.380bn (3.13%)   |
| VAT                       | €27.853bn (5.44%)   | €33.403bn (6.47%)   | €45.059bn (6.91%)   |
| Capital-formation taxes   | €4.561bn (0.89%)    | €18.704bn (3.62%)   | €23.254bn (3.57%)   |
| Export taxes              | €0.000bn (0.00%)    | €0.000bn (0.00%)    | €0.000bn (0.00%)    |
| Employee social insurance | €30.263bn (5.92%)   | €26.029bn (5.04%)   | €27.883bn (4.28%)   |
| Employer social insurance | €65.808bn (12.86%)  | €56.600bn (10.96%)  | €60.633bn (9.30%)   |
| Total revenue             | €218.161bn (42.64%) | €212.350bn (41.11%) | €257.927bn (39.55%) |

**Expenditure**

|                            | Period 1            | Period 20           | Period 50           |
|:---------------------------|:--------------------|:--------------------|:--------------------|
| Government consumption     | €148.587bn (29.04%) | €140.525bn (27.20%) | €170.108bn (26.09%) |
| Unemployment benefits      | €8.654bn (1.69%)    | €22.283bn (4.31%)   | €20.280bn (3.11%)   |
| Interest payments on debt  | €0.712bn (0.14%)    | €2.341bn (0.45%)    | €22.041bn (3.38%)   |
| Household social transfers | €198.258bn (38.75%) | €199.250bn (38.57%) | €221.515bn (33.97%) |
| Public pension benefits    | €133.939bn (26.18%) | €147.155bn (28.49%) | €167.415bn (25.67%) |
| Other social transfers     | €26.761bn (5.23%)   | €29.402bn (5.69%)   | €33.450bn (5.13%)   |
| Necessity support          | €28.903bn (5.65%)   | €0.411bn (0.08%)    | €0.370bn (0.06%)    |
| Total expenditure          | €347.556bn (67.93%) | €342.116bn (66.23%) | €413.664bn (63.43%) |

## Permanent-income decomposition

In [9]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)

fig = plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["real_pc_income_idx"],
    reducer="mean",
    title='real_pc_income_idx',
    show=True,
)


contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


,period,date,regressor,simulation_source,is_fixed,x_t,coefficient,contribution,point_forecast
0,0,2014Q1,time_trend,simulation_period_index,False,136.000000,0.004948,0.672896,0.022717
1,0,2014Q1,covid19,excluded_stage_3_dummy,False,0.000000,0.002059,0.000000,0.022717
2,0,2014Q1,log_real_pc_income,real_pc_income,False,4.605170,-1.017999,-4.688058,0.022717
3,0,2014Q1,d4_log_real_pc_income,real_pc_income,False,0.002550,0.068133,0.000174,0.022717
4,0,2014Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-0.757011,0.000638,-0.000483,0.022717
...,...,...,...,...,...,...,...,...,...
76,8,2016Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-2.202756,0.000638,-0.001405,0.050912
77,8,2016Q1,real_interest_rate_ma4_l5,policy_rate_and_cpi_fixed_basket,False,-0.487531,-0.002580,0.001258,0.050912
78,8,2016Q1,real_interest_rate_ma4_l9,policy_rate_and_cpi_fixed_basket,False,-0.996777,0.001678,-0.001673,0.050912
79,8,2016Q1,unemp_rate_ma4_l1,unemployment_rate,False,23.471861,-0.003852,-0.090413,0.050912


In [10]:
plot_agent_timeseries(
    model,
    "FRA",
    "households",
    variables = [
        'income',
        #  'non_property_income',
        'income_employee',
        # 'total_income_employee',
        'income_unemployment_benefits',
        'income_public_pension',
        'income_other_social_transfers',
        'income_social_transfers',
        # 'total_income_social_transfers',
        'income_rental',
        # 'total_income_rental',
        'income_dividend_distributions',
        # 'total_income_dividend_distributions'
        ],
    agg="mean",
    no_cols=3,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

In [11]:
# model.countries["FRA"].households.ts.get_keys()

## Validation

In [12]:
from src.validation import build_validation_availability_table, load_actual_policy_rate

# Availability check for the simulated-vs-actual comparison (autocorrelation and
# cross-correlation with real GDP), before any statistic is computed. See
# src/validation.py for series definitions, deflation, and rate-convention notes.
actual_policy_rate, policy_rate_available = load_actual_policy_rate(cfg, data)
validation_availability = build_validation_availability_table(data, df_scenario, COUNTRY, actual_policy_rate)
validation_availability

,simulated_column,simulated_available,actual_column,actual_available,both_available
series,,,,,
real_gdp,real_gdp,True,Real GDP (Value),True,True
real_consumption,household_consumption / ppi,True,Real Household Consumption (Value),True,True
real_investment,gfcf / ppi,True,Gross Fixed Capital Formation (Value) / PPI (V...,True,True
unemployment_rate,unemployment_rate,True,Unemployment Rate (Value),True,True
cpi,cpi_transaction,True,CPI (Value),True,True
central_bank_policy_rate,central_bank_policy_rate,True,"Policy Rate (BIS, quarterly, converted to per-...",True,True


In [13]:
from src.validation import build_validation_coverage_table, compute_validation_results

# User settings: how many lags to compute for each statistic.
MAX_ACF_LAGS = 12
MAX_CCF_LAGS = 8

validation_results, period_to_date = compute_validation_results(
    data, df_scenario, COUNTRY, actual_policy_rate, max_acf_lags=MAX_ACF_LAGS, max_ccf_lags=MAX_CCF_LAGS
)
build_validation_coverage_table(validation_results, n_total_periods=len(df_scenario.index))

real_gdp: actuals cover 38/51 model periods; statistics use the overlap only.
real_consumption: actuals cover 39/51 model periods; statistics use the overlap only.
real_investment: actuals cover 38/51 model periods; statistics use the overlap only.
unemployment_rate: actuals cover 29/51 model periods; statistics use the overlap only.
cpi: actuals cover 39/51 model periods; statistics use the overlap only.
central_bank_policy_rate: actuals cover 37/51 model periods; statistics use the overlap only.


,n_periods,n_total_periods
real_gdp,38,51
real_consumption,39,51
real_investment,38,51
unemployment_rate,29,51
cpi,39,51
central_bank_policy_rate,37,51


In [14]:
from src.validation import plot_validation_charts

# Charts: one row per validation variable, three panels each - level
# (simulated vs. actual on calendar years, dual y-axis), autocorrelation, and
# cross-correlation against real GDP.
plot_validation_charts(validation_results, period_to_date, COUNTRY).show()

## Sector and firm-credit diagnostics

In [15]:
# model.countries["FRA"].firms.ts.get_keys()

In [16]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_scenario)

credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)



### Investment
$$\text{total firm investment} = \text{capital-goods purchases} + \text{direct TFP spending} + \text{technical-investment spending},$$
$$\text{planned productivity investment} = \text{planned TFP investment} + \sum_{\text{inputs}}\text{planned technical investment}.$$

In [17]:
# Investment

investment_vars = [
    "total_capital_inputs_bought_costs",  # gross capital-goods purchases. (GFCF)
    # "net_capital_investment_above_replacement",  # gross capital purchases minus replacement/depreciation costs.
    # "planned_productivity_investment",  # total planned productivity investment.
    # "planned_tfp_investment",  # direct TFP component.
    # "planned_technical_investment",  # technical-coefficient component by input.
    "executed_tfp_investment",  # direct TFP investment actually executed.
    "executed_technical_investment",  # technical investment actually executed by input.
    ]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=investment_vars,
    agg="sum",
    no_cols=4,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

tfp_vars = [
    'tfp_investment_finance_scale',
    'planned_tfp_investment_expected_costs',
    'feasible_tfp_investment_expected_costs',
    'executed_productivity_investment',
    'net_capital_investment_above_replacement',
    'real_executed_productivity_investment',
    'planned_tfp_investment',
    'executed_tfp_investment',
    'direct_tfp_investment_cash_expense',
    'tfp_investment_effective_cost_rate',
    'tfp_investment_target_intensity',
    'tfp_investment_cap_intensity',
    'tfp_investment_desired_intensity',
    'tfp_investment_planned_intensity',
    'tfp_investment_desired_marginal_benefit',
    'tfp_investment_desired_marginal_cost',
    'tfp_investment_desired_mb_mc_ratio',
    'tfp_investment_desired_marginal_gap',
    'tfp_investment_planned_marginal_benefit',
    'tfp_investment_planned_marginal_cost',
    'tfp_investment_planned_mb_mc_ratio',
    'tfp_investment_planned_marginal_gap',
    'tfp_investment_cap_binding',
    'tfp_investment_cash_binding',
]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=tfp_vars,
    agg="sum",
    no_cols=4,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)

In [18]:
# Balance-sheet variables (point-in-time stocks/positions)
balance_sheet_vars = [
    # Inventory and working capital
    'inventory',
    'inventory_nominal',
    'unsold_output',
    'intermediate_inputs_stock',
    'intermediate_inputs_stock_value',
    # 'intermediate_inputs_stock_industry',
    'capital_inputs_stock',
    'capital_inputs_stock_value',
    # 'capital_inputs_stock_industry',
    'expected_capital_inputs_stock_value',
    
    # Equity
    'equity',
    
    # Debt
    'short_term_loan_debt',
    'long_term_loan_debt',
    'debt',
    # 'total_debt',
    # 'total_credit_exposure',
    
    # Deposits and cash
    'deposits',
    # 'total_deposits',
    
    # Accumulated arrears/balances
    'firm_settlement_opening_interest_arrears',
    'firm_settlement_closing_interest_arrears',
    'firm_settlement_opening_principal_arrears',
    'firm_settlement_closing_principal_arrears',
    'operating_revolving_opening_balance',
    'operating_revolving_closing_balance',
]

# Flow variables (transactions over a period)
flow_vars = [
    # Production and output
    'production',
    'production_nominal',
    'output_by_employee_histogram',
    'total_sales',
    'expected_sales',
    'real_amount_sold',
    'real_amount_sold_to_FRA',
    'real_amount_sold_to_ROW',
    'nominal_amount_sold_in_lcu',
    'nominal_amount_sold_in_lcu_to_FRA',
    'nominal_amount_sold_in_lcu_to_ROW',
    
    # Profitability and income
    'profits',
    'expected_profits',
    'gross_operating_surplus_mixed_income',
    'real_wage_per_capita',
    
    # Costs and expenses
    'total_wage',
    'unit_costs',
    'labour_costs',
    'used_intermediate_inputs',
    'used_intermediate_inputs_costs',
    'total_intermediate_inputs_bought_costs',
    'used_capital_inputs',
    'used_capital_inputs_costs',
    'capital_depreciation_costs',
    'total_capital_inputs_bought_costs',
    
    # Taxes
    'taxes_paid_on_production',
    'corporate_taxes_paid',
    
    # Debt service (interest and principal)
    'interest_paid_on_deposits',
    'interest_paid_on_loans',
    'interest_paid',
    'firm_settlement_scheduled_interest_due',
    'firm_settlement_contractual_interest_due',
    'firm_settlement_payable_interest',
    'firm_settlement_unpaid_interest',
    'firm_settlement_capitalized_interest',
    'operating_revolving_interest_base',
    'operating_revolving_interest_paid',
    'debt_installments',
    'scheduled_debt_service',
    'total_debt_installments',
    'firm_settlement_scheduled_principal_due',
    'firm_settlement_contractual_principal_due',
    'firm_settlement_payable_principal',
    'firm_settlement_unpaid_principal',
    
    # Investment
    'planned_productivity_investment',
    'executed_productivity_investment',
    'net_capital_investment_above_replacement',
    'real_executed_productivity_investment',
    'planned_technical_investment',
    'executed_technical_investment',
    'technical_investment_by_input',
    'gross_fixed_capital_formation',
    'planned_tfp_investment',
    'executed_tfp_investment',
    'direct_tfp_investment_cash_expense',
    
    # Labour flows
    'labour_inputs',
    'desired_labour_inputs',
    'number_of_employees',
    
    # Demand and purchases
    'demand',
    'estimated_demand',
    'real_excess_demand',
    'accepted_goods_order',
    'goods_order',
    'real_amount_bought',
    'real_amount_bought_from_FRA',
    'real_amount_bought_from_ROW',
    'real_amount_bought_as_intermediate_inputs',
    'real_amount_bought_as_capital_goods',
    
    # Spending
    'nominal_amount_spent_in_usd',
    'nominal_amount_spent_in_usd_to_FRA',
    'nominal_amount_spent_in_usd_to_ROW',
    'nominal_amount_spent_in_lcu',
    'nominal_amount_spent_in_lcu_to_FRA',
    'nominal_amount_spent_in_lcu_to_ROW',
    
    # Inventory changes
    'total_inventory_change',
    
    # Dividends
    'dividend_fund_declared_distribution',
    
    # Credit received
    'received_short_term_credit',
    'total_received_short_term_credit',
    'received_debt_rollover_credit',
    'total_received_debt_rollover_credit',
    'received_overdraft_refinance_credit',
    'total_received_overdraft_refinance_credit',
    'received_operating_refinance_credit',
    'total_received_operating_refinance_credit',
    'received_ordinary_short_term_credit',
    'total_received_ordinary_short_term_credit',
    'received_long_term_credit',
    'total_received_long_term_credit',
    
    # Wage growth
    'wage_offer_growth',
    'wage_offer_growth_historic_wage',
    'wage_offer_growth_tfp',
    'wage_offer_growth_productivity',
    'wage_offer_growth_tightness',
    'wage_offer_growth_residual',
]

plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=balance_sheet_vars,
    agg="sum",
    no_cols=3,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)

## Household Diagnostics

In [19]:
hh_vars = [
    "total_target_consumption_loans",
    [
        "target_consumption",
        "amount_bought",
        "consumption",
        "total_consumption",
    ],
    ["target_investment", "total_investment"],
    [
        "expected_income",
        "income",
    ],
    ["consumption_loan_debt", "total_consumption_loan_debt", "received_consumption_loans"],
    "liquidity_shortfall",
    "household_saving",
    "total_mortgage_debt",
    "portfolio_actual_illiquid_share",
    "portfolio_target_tfa_base",
    "portfolio_post_return_lfa",
    "portfolio_post_return_ifa",
    "portfolio_liquid_return_rate",
    "portfolio_illiquid_return_rate",
    "portfolio_investable_surplus",
    "portfolio_target_illiquid_share",
    "portfolio_target_illiquid_assets",
]


from src.visual_helpers import plot_agent_timeseries  # noqa: E402

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    hh_vars,
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    # no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [20]:
from src.visual_helpers import plot_agent_timeseries

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["target_consumption", "consumption", "amount_bought"],
        [
            "wealth_real_assets",
            "wealth_main_residence",
            "wealth_other_properties",
            "wealth_other_real_assets",
            "wealth_deposits",
            "wealth_other_financial_assets",
            "wealth_financial_assets",
        ],
        ["wealth", "net_wealth", "debt"],
        [
            "expected_income",
            "expected_income_employee",
            "expected_income_social_transfers",
            "expected_income_financial_assets",
        ],
        ["income", "income_employee", "income_social_transfers", "income_rental", "income_financial_assets"],
        ["target_investment", "investment", "total_investment", "total_investment_before_vat"],
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

ValueError: households.ts has no field 'wealth_deposits'.

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["mortgage_debt", "consumption_loan_debt", "debt"],
        ["target_mortgage", "received_mortgages", "received_consumption_loans"],
        ["debt_installments", "interest_paid_on_deposits", "interest_paid_on_loans", "interest_paid"],
        [
            "real_amount_sold",
            "real_amount_sold_to_FRA",
            "real_amount_sold_to_ROW",
            "nominal_amount_sold_in_lcu",
            "nominal_amount_sold_in_lcu_to_FRA",
            "nominal_amount_sold_in_lcu_to_ROW",
        ],
        [
            "nominal_amount_spent_in_usd",
            "nominal_amount_spent_in_usd_to_FRA",
            "nominal_amount_spent_in_usd_to_ROW",
            "nominal_amount_spent_in_lcu",
            "nominal_amount_spent_in_lcu_to_FRA",
            "nominal_amount_spent_in_lcu_to_ROW",
        ],
        ["real_amount_bought", "real_amount_bought_from_FRA", "real_amount_bought_from_ROW"],
    ],
    agg="sum",
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    ["price_paid_for_property", ["rent", "rent_imputed"], "max_price_willing_to_pay", "max_rent_willing_to_pay"],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["mortgage_debt", "consumption_loan_debt"],
        "price_paid_for_property",
        "wealth_real_assets",
        "wealth_main_residence",
        "wealth_other_properties",
        "wealth_other_real_assets",
        "wealth_deposits",
        "wealth_other_financial_assets",
        "wealth_financial_assets",
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=4,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=400,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        "income_rental",
        "wealth_real_assets",
        "wealth_main_residence",
        "wealth_other_properties",
        "wealth_other_real_assets",
        "wealth_deposits",
        "wealth_other_financial_assets",
        "wealth_financial_assets",
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=4,
    show_legend=True,
    show=False,
    # condition="wealth_other_financial_assets<0",
    agent_id=["5169"],
    base_height=300,
    base_width=400,
)

fig.show()


## Optional Monte Carlo

In [ ]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-1,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")


In [20]:
# Verify consistency ALM PLM

sigma_2_v_min = 0.0001
sigma_2_v_max = 0.0013
sigma_2_v_max = 0.0005
rho = 0.951
sigma_2_eta_a = 0.029
sigma_2_beta_a = 0.00038

sigma_2_eta_q = (1-rho**2) * sigma_2_eta_a / (1- rho**8) 
# print(sigma_2_eta_q**(1/2))
# print(sigma_2_beta_a / 16)
print(sigma_2_eta_q)

sigma_2_eta_implied_min = (sigma_2_v_min - (1 - rho)**2 * sigma_2_beta_a) / (rho**4 - 1)**2
print(sigma_2_eta_implied_min)

sigma_2_eta_implied_max = (sigma_2_v_max - (1 - rho)**2 * sigma_2_beta_a) / (rho**4 - 1)**2
print(sigma_2_eta_implied_max)



0.008376445599508184
0.002989481681828435
0.015057514729058492
